In [117]:
import numpy as np
import pandas as pd

In [118]:
train = np.load("C:/Users/colin/Desktop/Repos/Python Repo/Neural Networks/Lab4/train.npz",allow_pickle=True)
test = np.load("C:/Users/colin/Desktop/Repos/Python Repo/Neural Networks/Lab4/test.npz",allow_pickle=True)


In [119]:
train_data = train["data"] / 255.0
train_data = np.array(train_data)

train_labels = train["labels"]
train_labels = np.array(train_labels)

test_data = test["data"] / 255.0
test_data = np.array(test_data)

In [120]:
print("Full train data:", train_data.shape)
print("Full train labels:", train_labels.shape)
print("Test data:", test_data.shape)

Full train data: (60000, 784)
Full train labels: (60000,)
Test data: (15000, 784)


In [121]:
X_train = train_data.T
Y_train = train_labels
X_test = test_data.T

In [122]:
print("Training split:", X_train.shape)
print("Validation split:", Y_train.shape)

Training split: (784, 60000)
Validation split: (60000,)


In [123]:
def init_params():
    W = np.random.randn(10,784) * np.sqrt(2.0 / 784)
    b = np.zeros((10,1))
    return W,b

def softmax(Z):
    Z_shift = Z - np.max(Z, axis = 0, keepdims = True)
    exp_Z = np.exp(Z_shift)
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True) 

def forward_prop(W,b,X):
    Z = W.dot(X) + b
    A = softmax(Z)
    return Z,A

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, 10))
    one_hot_Y[np.arange(Y.size,),Y] = 1
    return one_hot_Y.T

def back_prop(Z,A,W,X,Y):
    m = Y.size
    
    one_hot_Y = one_hot(Y)
    
    dZ = A - one_hot_Y
    dW = 1 / m * dZ.dot(X.T)
    db = 1 / m * np.sum(dZ,axis=1, keepdims=True)

    return dW, db

def update_params(W,b,dW,db,alpha):
    W = W - alpha * dW
    b = b - alpha * db
    return W,b

def get_prediction(A):
    return np.argmax(A,0)

def get_accuracy(predictions,Y):
    return np.sum(predictions == Y) / Y.size

In [124]:
def gradient_descent(X,Y,iterations, alpha):
    W,b = init_params()
    for i in range(iterations):
        Z,A = forward_prop(W,b,X)
        dW, db = back_prop(Z,A,W,X,Y)
        W, b = update_params(W,b,dW,db,alpha)
        if i%10 == 0:
            print("Iteration: ", i)
            print("Accuracy: ", get_accuracy(get_prediction(A),Y))
    return W1,b1,W2,b2

def gradient_descent_experimental(X,Y,iterations, alpha, batch_size):
    W,b = init_params()
    m = X.shape[1]
    for i in range(iterations):
        perm = np.random.permutation(m)
        X_shuffled = X[:, perm]
        Y_shuffled = Y[perm]
        for j in range(0,m,batch_size):
            X_batch = X_shuffled[:,j:j+batch_size]
            Y_batch = Y_shuffled[j:j+batch_size]
            
            Z,A = forward_prop(W,b,X_batch)
            dW, db = back_prop(Z,A,W,X_batch,Y_batch)
            W, b = update_params(W,b,dW,db,alpha)
            
        if i%10 == 0 or i == iterations - 1:
            _,A2a = forward_prop(W,b,X)
            print("Iteration: ", i)
            print("Accuracy: ", get_accuracy(get_prediction(A2a),Y))
    return W,b

#W1, b1, W2, b2 = gradient_descent_experimental(X_train, Y_train, 100, 0.1,64)
#W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 100, 0.1)
W,b = gradient_descent_experimental(X_train, Y_train, 100, 0.1,64)

Iteration:  0
Accuracy:  0.79665
Iteration:  10
Accuracy:  0.8957333333333334
Iteration:  20
Accuracy:  0.9086
Iteration:  30
Accuracy:  0.9136333333333333
Iteration:  40
Accuracy:  0.91855
Iteration:  50
Accuracy:  0.9212333333333333
Iteration:  60
Accuracy:  0.9248166666666666
Iteration:  70
Accuracy:  0.9269166666666667
Iteration:  80
Accuracy:  0.9282333333333334
Iteration:  90
Accuracy:  0.9300166666666667
Iteration:  99
Accuracy:  0.9309666666666667


In [125]:
def make_prediction(X,W,b):
    _, A = forward_prop(W,b,X)
    return get_prediction(A)

In [126]:
test_predictions = make_prediction(X_test,W,b)

In [127]:
submission = pd.DataFrame({
    "ID": range(len(test_predictions)),
    "target": test_predictions,
})

In [128]:
submission.to_csv("submission2.csv", index=False)
print("Created submission.csv")

Created submission.csv
